# Mosna Analysis

In [1]:
import warnings
from sklearn.exceptions import ConvergenceWarning, FitFailedWarning
warnings.simplefilter('ignore', FitFailedWarning)
warnings.simplefilter('ignore', ConvergenceWarning)
warnings.simplefilter('ignore', FutureWarning)
warnings.simplefilter('ignore', DeprecationWarning)
warnings.simplefilter('ignore', UserWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from time import time
import warnings
import joblib
from pathlib import Path
from time import time
from tqdm import tqdm
import copy
import matplotlib as mpl
import napari
import colorcet as cc
import composition_stats as cs
from sklearn.impute import KNNImputer
from lifelines import KaplanMeierFitter, CoxPHFitter

from tysserand import tysserand as ty
from mosna import mosna

import matplotlib as mpl
mpl.rcParams["figure.facecolor"] = 'white'
mpl.rcParams["axes.facecolor"] = 'white'
mpl.rcParams["savefig.facecolor"] = 'white'

In [2]:
import glob
import re

objects_path = glob.glob("data/processed/Bram_IMC/acquired_csv_files/*.csv")
files=dict()



sample_number=1
for file in objects_path:
    name_file = Path(file).name
    name_file = re.split(r'[_.-]', name_file)
    name_file = name_file[1:3]
    if Path(file).with_suffix('.parquet').exists():
        obj = pd.read_parquet(Path(file).with_suffix('.parquet'))
        obj.drop(columns='Unnamed: 0', inplace=True)
        obj['patient']=name_file[0]
        obj['sample']=name_file[1]
        files.setdefault(f'sample {sample_number}', obj)
    else:
        obj = pd.read_csv(Path(file))
        obj.to_parquet(Path(file).with_suffix('.parquet'))
        obj.drop(columns='Unnamed: 0', inplace=True)
        obj['patient']=name_file[0]
        obj['sample']=name_file[1]
        files.setdefault(f'sample {sample_number}', obj)
    
    sample_number+=1
        
    

In [3]:
verif = files["sample 1"].columns
concatanable = True

for file in files.values():
    
    if file.columns.all() != verif.all():
        print("not concatanable files")
        concatanable = False

nb_row=0
nb_row_concat=0
if concatanable:
    concat_obj=pd.DataFrame()
    for file in files.values():
        nb_row+=file.shape[0]
        concat_obj = pd.concat([concat_obj, file], ignore_index=True)
    nb_row_concat=concat_obj.shape[0]

if nb_row == nb_row_concat:
    print(f"Concaténation réussite, nombre de row : {nb_row_concat}")



Concaténation réussite, nombre de row : 401568


In [4]:
### sample n° --> cell-ID 
cell_ID = pd.Series([i for i in range(len(concat_obj))], name='cell_ID')
sample_cell = pd.concat([cell_ID,concat_obj[['patient','sample']]],axis=1)
sample_cell

,cell_ID,patient,sample
0,0,16,01
1,1,16,01
2,2,16,01
3,3,16,01
4,4,16,01
...,...,...,...
401563,401563,09,03
401564,401564,09,03
401565,401565,09,03
401566,401566,09,03


In [5]:
concat_obj

,aSMA_Cd110,CD3_Sm152,CD4_Gd156,CD8_Dy162,CD11b_Sm149,CD11c_Sm154,CD14_Nd144,CD15_Y89,CD16_Nd146,CD20_Nd142,...,Lag3_Eu153,PanCK_Nd148,PD1_Ho165,PDL1_Lu175,Tim3_Yb171,Vim_Nd143,X_position,Y_position,patient,sample
0,0.670535,0.773695,0.412637,1.925640,3.060393,0.670535,0.395444,5.759728,0.103159,0.567376,...,0.034386,0.051580,0.395444,0.017193,1.822481,0.429830,18.241379,4.396552,16,01
1,0.478659,2.313519,0.239330,1.435978,0.159553,0.079777,0.797765,3.988827,0.159553,0.119665,...,0.199441,0.358994,0.239330,0.000000,4.866368,1.356201,26.440000,2.880000,16,01
2,0.295469,0.184668,0.627871,0.295469,0.295469,0.775605,1.809745,9.122594,0.664804,0.517070,...,0.110801,0.258535,0.036934,0.258535,1.957480,0.738672,34.518520,2.962963,16,01
3,2.466774,0.104969,0.104969,0.104969,0.209938,0.262423,0.629815,3.516465,0.209938,0.104969,...,0.000000,0.052485,0.000000,0.052485,1.994413,1.102176,76.842100,1.789474,16,01
4,0.073867,0.313935,0.313935,0.258535,0.960273,0.295469,3.508690,6.740378,0.110801,0.166201,...,0.092334,2.825419,0.110801,0.018467,6.740378,0.554004,93.074070,3.203704,16,01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401563,0.034921,0.680967,1.327012,0.279371,0.296832,0.034921,1.012720,0.069843,0.104764,0.192068,...,0.034921,0.087303,0.174607,0.034921,4.836610,0.803191,2419.544000,1459.491200,09,03
401564,0.100968,0.158664,0.317329,0.158664,0.187513,0.072120,0.865443,0.230785,0.375025,0.230785,...,0.014424,0.144240,0.072120,0.043272,2.120334,1.052955,2503.869600,1460.275400,09,03
401565,0.142180,0.236966,0.236966,0.379146,1.563978,0.000000,2.796204,0.189573,0.616113,0.142180,...,0.000000,0.094787,0.142180,0.094787,1.753552,1.753552,2567.476000,1460.809600,09,03
401566,0.122493,0.428727,0.214363,0.244987,1.255558,0.336857,5.450957,0.137805,0.704337,0.244987,...,0.045935,0.122493,0.061247,0.030623,4.011660,0.842142,2583.015400,1459.384600,09,03


In [6]:
cell_pos=pd.concat([sample_cell['cell_ID'],concat_obj[['X_position','Y_position']]],axis=1)
concat_obj.drop(columns=['X_position','Y_position','patient','sample'], inplace=True)
cell_pos

,cell_ID,X_position,Y_position
0,0,18.241379,4.396552
1,1,26.440000,2.880000
2,2,34.518520,2.962963
3,3,76.842100,1.789474
4,4,93.074070,3.203704
...,...,...,...
401563,401563,2419.544000,1459.491200
401564,401564,2503.869600,1460.275400
401565,401565,2567.476000,1460.809600
401566,401566,2583.015400,1459.384600


In [7]:
del files, verif, nb_row, nb_row_concat, sample_number, concatanable, cell_ID, file, name_file, obj, objects_path